In [ ]:
import requests
from shapely.geometry import Point, shape
import shapely
import geojson
import json
from pprint import pprint
from shapely.ops import unary_union
from shapely.geometry import Point
from shapely.strtree import STRtree
import pickle


OVERPASS_URL = "https://overpass-api.de/api/interpreter"

# Countries

### Extract borders of all countries

In [179]:
with open('ALL-high-res.json', encoding='utf-8') as f:
    data = json.load(f)

In [180]:
result = {}
for feature in data["features"]:
    iso = feature["properties"]["iso_a2_eh"]

    if iso == '-99':
        continue

    if iso not in result:
        result[iso] = {
            "name": None,
            "geoms": [],
            "poly": None
        }

    name = feature["properties"]["name_ciawf"]
    if result[iso]["name"] is None and name is not None:
        result[iso]["name"] = name

    poly = shapely.from_geojson(json.dumps(feature["geometry"]))
    result[iso]["geoms"].append(poly)

for key, value in result.items():
    if len(value["geoms"]) > 0:
        value["poly"] = unary_union(value["geoms"])
    else:
        value["poly"] = value["geoms"]

with open("countries.pkl", "wb") as f:
    pickle.dump(result, f)


# OpenStreetmap

## Fetch cities

In [ ]:
# query = """
# [out:json][timeout:180];
# (
#   node[place~"city|town|village"](around:3000,49.440358,6.07191);
# );
# out center;
# """

query = """
[out:json][timeout:180];

node
  [place~"^(city|town|village)$"]
  (around:300000,49.440358,6.07191);
out ids tags geom;
"""
response = requests.post(OVERPASS_URL, data={"data": query})
data = response.json()

cities = [
    {
        "name": elem["tags"].get("name", "unknown"),
        "type": elem["tags"].get("place"),
        "lat": elem["lat"],
        "lon": elem["lon"],
        "id":  elem["id"],
    }
    for elem in data["elements"]
]

for city in cities[:10]:
    print(city)

{'name': 'Charleroi', 'type': 'city', 'lat': 50.4116233, 'lon': 4.444528, 'id': 9002746}
{'name': 'Raidelbach', 'type': 'village', 'lat': 49.709349, 'lon': 8.7352499, 'id': 11437285}
{'name': 'Reichenbach', 'type': 'village', 'lat': 49.7129657, 'lon': 8.6930453, 'id': 12101529}
{'name': 'Elmshausen', 'type': 'village', 'lat': 49.7016047, 'lon': 8.6722226, 'id': 12101530}
{'name': 'Wilmshausen', 'type': 'village', 'lat': 49.6957672, 'lon': 8.6620534, 'id': 12101531}
{'name': 'Schönberg', 'type': 'village', 'lat': 49.6943101, 'lon': 8.6489156, 'id': 12101532}
{'name': 'Bensheim', 'type': 'town', 'lat': 49.6810158, 'lon': 8.6227577, 'id': 12101533}
{'name': 'Wendlingen am Neckar', 'type': 'town', 'lat': 48.6727602, 'lon': 9.3838404, 'id': 13354924}
{'name': 'Esslingen am Neckar', 'type': 'town', 'lat': 48.7427584, 'lon': 9.3071685, 'id': 13355228}
{'name': 'Plochingen', 'type': 'town', 'lat': 48.711173, 'lon': 9.4184961, 'id': 13675911}


## Include country

In [ ]:
# 1. Vos données (exemples)
country_name, country_poly = [], []
with open("countries.pkl", "rb") as f:
    result = pickle.load(f)

for country in result.values():
    country_name.append(country["name"])
    country_poly.append(country["poly"])

# 2. Créer l'index spatial sur les pays
tree = STRtree(country_poly)

# 3. Créer les points Shapely
points = [Point(city["lon"], city["lat"]) for city in cities]

# 4. Requête de masse : qui est à l'intérieur de quoi ?
# query_bulk retourne deux tableaux d'indices : [indices_points, indices_pays]
indices_points, indices_pays = tree.query(points, predicate='within')

# 5. Associer les résultats
for idx_pt, idx_country in zip(indices_points, indices_pays):
    cities[idx_pt]["country"] = country_name[idx_country]

In [195]:
for city in cities:
    if "udange" in city["name"].lower():
        print(city["name"])

Udange
Azoudange
Haboudange
Hombourg-Budange
Budange


In [196]:
with open("cities.pkl", "wb") as f:
    pickle.dump(cities, f)

In [197]:
cities[:2]

[{'name': 'Charleroi',
  'type': 'city',
  'lat': 50.4116233,
  'lon': 4.444528,
  'id': 9002746,
  'country': 'Belgium'},
 {'name': 'Raidelbach',
  'type': 'village',
  'lat': 49.709349,
  'lon': 8.7352499,
  'id': 11437285,
  'country': 'Germany'}]

In [204]:
print(set(x["country"] for x in cities if "country" in x))

{'Germany', 'Luxembourg', 'Belgium', 'France', 'Switzerland', 'Netherlands'}


In [203]:
for city in cities:
    if not "country" in city:
        print(city)

{'name': 'Ellewoutsdijk', 'type': 'village', 'lat': 51.3896185, 'lon': 3.8160901, 'id': 1930294883}
{'name': 'Herkingen', 'type': 'village', 'lat': 51.7103863, 'lon': 4.0873287, 'id': 1941875674}
{'name': 'Hansweert', 'type': 'village', 'lat': 51.4490517, 'lon': 4.0037349, 'id': 2005027736}
{'name': 'Sint Philipsland', 'type': 'village', 'lat': 51.6169203, 'lon': 4.1663627, 'id': 2526590373}
{'name': 'Vlissingen', 'type': 'town', 'lat': 51.4480929, 'lon': 3.5697992, 'id': 3122829496}
{'name': 'Zoutelande', 'type': 'village', 'lat': 51.5018348, 'lon': 3.4872245, 'id': 3130166299}
{'name': 'Westkapelle', 'type': 'village', 'lat': 51.5293313, 'lon': 3.4407863, 'id': 3130176450}
{'name': 'Borssele', 'type': 'village', 'lat': 51.4254902, 'lon': 3.7379392, 'id': 6627669368}
